# 14.6 - Memory

Status: VERIFIED

## What Are We Solving?
Without memory, every interaction starts from zero. Memory enables personalization, context accumulation, and continuity across long conversations.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Key-Value Memory Store

In [2]:
import time

class MemoryStore:
    def __init__(self):
        self.facts = []
    
    def save(self, fact: str, category: str = "general"):
        self.facts.append({
            "fact": fact,
            "category": category,
            "timestamp": time.time()
        })
        print(f"  Saved: [{category}] {fact[:60]}")
    
    def search(self, query: str, top_k: int = 5) -> list:
        query_words = set(query.lower().split())
        scored = []
        for entry in self.facts:
            fact_words = set(entry["fact"].lower().split())
            overlap = len(query_words & fact_words)
            scored.append((overlap, entry))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [entry for _, entry in scored[:top_k]]
    
    def recent(self, n: int = 5) -> list:
        return self.facts[-n:]

memory = MemoryStore()
memory.save("User prefers Python over JavaScript", "preference")
memory.save("User is building a RAG system", "context")
memory.save("User's project uses FastAPI", "context")
memory.save("User wants concise answers", "preference")

print("\nSearch for 'preference':")
for fact in memory.search("preference"):
    print(f"  - {fact['fact']}")

  Saved: [preference] User prefers Python over JavaScript
  Saved: [context] User is building a RAG system
  Saved: [context] User's project uses FastAPI
  Saved: [preference] User wants concise answers

Search for 'preference':
  - User prefers Python over JavaScript
  - User is building a RAG system
  - User's project uses FastAPI
  - User wants concise answers


## Agent with Memory

In [3]:
def agent_with_memory(user_input: str, memory: MemoryStore) -> str:
    """Agent that loads relevant memory before responding."""
    # Retrieve relevant memories
    relevant = memory.search(user_input, top_k=3)
    memory_context = "\n".join([f"- {m['fact']}" for m in relevant])
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                f"Relevant context from memory:\n{memory_context}\n\n"
                "Use this context to personalize your response. Be concise."
            )},
            {"role": "user", "content": user_input}
        ]
    )
    
    answer = response.choices[0].message.content
    
    # Save new information
    memory.save(f"User asked: {user_input[:50]}", "interaction")
    
    return answer

# Test
answer = agent_with_memory("What should I use for my project?", memory)
print(f"Answer: {answer[:200]}")
print(f"\nMemory now has {len(memory.facts)} facts")

  Saved: [interaction] User asked: What should I use for my project?
Answer: Given your context, you should stick with your current tech stack:

1. **Python** – Aligns with your preference and dominates the AI/ML ecosystem for RAG systems.
2. **FastAPI** – Solid choice for ser

Memory now has 5 facts


In [4]:
# Verification
assert len(memory.facts) >= 4, "Must have saved facts"
assert len(memory.search("python")) > 0, "Search must work"
print("VERIFICATION PASSED: Phase 14.6 complete")

VERIFICATION PASSED: Phase 14.6 complete
